In [10]:
import math
import pandas as pd
import numpy as np
import tensorflow as tf
import utils as utils
import dataset as ds

config = utils.load_config("./configs/config.yaml")

NUM_SHARDS = config['data']['num_shards']
BATCH_SIZE = config['training']['batch_size']

SEED = config['seed']

tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# 2) 가능하면 TF 연산 deterministic
# tf.config.experimental.enable_op_determinism()

# 가장 먼저 실행해야 함
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU {len(gpus)}개 사용 중")
    
tf.keras.mixed_precision.set_global_policy("mixed_float16")

if config['train_finetune'] == False:
    train_data_path = config['data']['path'] + config['data']['train_path']
    #val_data_path = config['data']['path'] + config['data']['val_path']
    
    data = pd.read_csv(train_data_path, encoding='utf-8-sig')
    
    #val_data = pd.read_csv(val_data_path, encoding='utf-8-sig')
    #data = pd.concat([train_data, val_data])
    
    data.dropna(inplace=True)
    data = data[(data['match_ratio'] > 0.7)]
else:
    finetune_data_path = r"C:\Users\user\Downloads\수어 영상\1.Training\finetune.csv"
    data = pd.read_csv(finetune_data_path, encoding='utf-8-sig')

#학습 데이터에 존재하는 단어를 인덱싱
#vocabulary = utils.Vocabulary(data['morpheme'])
#vocabulary.save_csv(config['vocabulary']['path'])
vocabulary = utils.Vocabulary.load_csv(config['vocabulary']['path'])

#모든 단어들을 인덱스 번호로 전환
data['morpheme'] = data['morpheme'].apply(
    lambda m: np.array(vocabulary.encode(m), dtype=np.int32)
)

#데이터 셔플
data = data.sample(frac=1, random_state=SEED).reset_index(drop=True)

shard_datasets = ds.make_shard_dataset(data, NUM_SHARDS)

GPU 1개 사용 중


In [11]:
from model import make_model

sign_model = make_model(len(vocabulary.vocab)) 

init_lr = config['optimizer']['init_lr']
min_lr = config['optimizer']['min_lr']
decay_rate = config['optimizer']['decay_rate']
dynamic_growth_steps = config['optimizer']['dynamic_growth_steps']

base_opt = tf.keras.optimizers.Adam(init_lr, clipnorm=1.0)
optimizer = tf.keras.mixed_precision.LossScaleOptimizer(
    base_opt, 
    dynamic=True,
    initial_scale=2**7,
    dynamic_growth_steps=dynamic_growth_steps
)

checkpoint = tf.train.Checkpoint(
    model=sign_model,
    optimizer=optimizer,
    shard_idx=tf.Variable(0),
    epoch=tf.Variable(0),
    step=tf.Variable(0),
    global_step=tf.Variable(0),
    skip_step_count_in_epoch=tf.Variable(0),
    total_loss=tf.Variable(0.0, dtype=tf.float32),
)

sign_model.compile(
    optimizer=optimizer,
    loss=None
)

manager = tf.train.CheckpointManager(
    checkpoint,
    directory=config['training']['checkpoint']['directory'],
    max_to_keep=config['training']['checkpoint']['max_to_keep']
)

sign_model.summary(expand_nested=True, show_trainable=True)

@tf.function(
    input_signature=[
        tf.TensorSpec([None, None, 224, 224, 3], tf.float32),
        tf.TensorSpec([None, None, 381], tf.float32),
        tf.TensorSpec([None], tf.int32),
        tf.TensorSpec([None, None], tf.int32),
    ],
    reduce_retracing=True,
)
def train_step(video, keypoint, video_length, morpheme):
    # 입력 유한성 체크
    tf.debugging.assert_all_finite(video, "video NaN/Inf")
    tf.debugging.assert_all_finite(keypoint, "keypoint NaN/Inf")
    
    with tf.GradientTape() as tape:
        logits = sign_model([video, keypoint], training=True)
        
        # #수치 안정화
        #logits = tf.cast(logits, tf.float32)
        logits = logits / 2.0
        logits = tf.clip_by_value(logits, -20.0, 20.0)

        logit_length = tf.cast(video_length, tf.int32)
        label_length = tf.reduce_sum(
            tf.cast(tf.not_equal(morpheme, 3), tf.int32), axis=1
        )
        
        # CTC 길이 조건
        tf.debugging.assert_less_equal(label_length, logit_length)
        
        labels_for_ctc = tf.where(tf.equal(morpheme, 3), tf.zeros_like(morpheme), morpheme)
        
        tf.debugging.assert_less_equal(label_length, logit_length)
        tf.debugging.assert_greater_equal(tf.reduce_min(labels_for_ctc), 0)
        tf.debugging.assert_less(tf.reduce_max(labels_for_ctc), tf.shape(logits)[-1])

        loss = tf.nn.ctc_loss(
            labels=labels_for_ctc,
            logits=tf.transpose(logits, [1, 0, 2]),
            label_length=label_length,
            logit_length=logit_length,
            blank_index=0,
            logits_time_major=True
        )
        
        loss = tf.reduce_mean(loss)
        
        # logits, loss finite 체크
        logits_finite = tf.reduce_all(tf.math.is_finite(logits))
        loss_finite = tf.reduce_all(tf.math.is_finite(loss))
        should_apply = tf.logical_and(logits_finite, loss_finite)
        
        # skip일 때도 그래프 유지 위해 0-loss 사용
        safe_loss = tf.where(should_apply, loss, tf.zeros_like(loss))
        scaled_loss = optimizer.get_scaled_loss(safe_loss)
        
    scaled_grads = tape.gradient(scaled_loss, sign_model.trainable_variables)
    grads = optimizer.get_unscaled_gradients(scaled_grads)
    
    clipped_grads, grad_norm = tf.clip_by_global_norm(grads, 1.0)
    
    def _apply(): 
        optimizer.apply_gradients(zip(clipped_grads, sign_model.trainable_variables))
        return tf.constant(False)  # skipped=False

    def _skip():
        return tf.constant(True)   # skipped=True
    
    skipped = tf.cond(should_apply, _apply, _skip)
    
    # 로그용
    all_finite_grad = tf.reduce_all([
        tf.reduce_all(tf.math.is_finite(g)) for g in grads if g is not None
    ])

    return tf.cast(loss, tf.float32), all_finite_grad, grad_norm, skipped

Model: "model_1"
_____________________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     Trainable  
 input_5 (InputLayer)           [(None, None, 381)]  0           []                               Y          
                                                                                                             
 input_4 (InputLayer)           [(None, None, 224,   0           []                               Y          
                                224, 3)]                                                                     
                                                                                                             
 lstm_1 (LSTM)                  (None, None, 128)    261120      ['input_5[0][0]']                Y          
                                                                                                       

In [12]:
def create_result(path):
    result = pd.DataFrame(columns=['epoch', 'train_avg_loss', 'val_macro_wer', 'val_micro_wer'])
    result.to_csv(path, encoding='utf-8-sig', index=False)

In [14]:
#파인튜닝에 사용될 가중치 설정
if config['train_finetune'] == True:
    sign_model.load_weights(r".\model_weight\sign_model(new_normalize)_15.weights.h5")

In [15]:
import os
from pathlib import Path
from tqdm import tqdm
import logging

config = utils.load_config("./configs/config.yaml")

SAVE_RESULT = False

if config['resume_train'] == True:
    print("Train Resume")
    checkpoint.restore(manager.latest_checkpoint)
    
logging.basicConfig(
    filename=config['logging']['directory'],
    level=logging.INFO,
    encoding='utf-8'
)

if not os.path.exists(config['training']['result']['path']):
    result_path = Path(config['training']['result']['path'])
    result_path.parent.mkdir(parents=True, exist_ok=True)
    create_result(config['training']['result']['path'])

if SAVE_RESULT == True:
    result = pd.read_csv(config['training']['result']['path'])

TRAINING_EPOCH = config['training']['epoch']

start_epoch = int(checkpoint.epoch.numpy())

# 전체 step 계산
total_step = 0
for shard_idx in range(0, len(shard_datasets)):
    shard_len = len(shard_datasets[shard_idx])
    shard_steps =  math.ceil(shard_len / BATCH_SIZE)
    total_step += shard_steps

for epoch in range(start_epoch, TRAINING_EPOCH):
    cur_step = int(checkpoint.step.numpy())
    cur_shard_idx = int(checkpoint.shard_idx.numpy())
    global_step = int(checkpoint.global_step.numpy())
    skip_step_count_in_epoch = int(checkpoint.skip_step_count_in_epoch.numpy())
    total_loss = float(checkpoint.total_loss.numpy())
            
    pgBar = tqdm(total=total_step, initial=global_step, desc=f"Epoch {epoch+1}", unit="step")
    
    for shard_idx in range(cur_shard_idx, len(shard_datasets)):
        shard_dataset = ds.make_dataset_from_df(shard_datasets[shard_idx], BATCH_SIZE)
        skip_step_count = 0
        
        for step, ((video, keypoint, video_length), morpheme) in enumerate(shard_dataset):
            
            #cur_step이 넘을 때까지 건너뛰기
            if shard_idx == cur_shard_idx and step < cur_step:
                continue
            
            #is_all_finite, loss, all_finite_grad, grad_norm = train_step(video, keypoint, video_length, morpheme)
            loss, all_finite_grad, grad_norm, skipped = train_step(video, keypoint, video_length, morpheme)
            
            if (not bool(all_finite_grad.numpy())) or bool(skipped.numpy()):
                    logging.warning(
                        f"[WARN] epoch= {epoch + 1} step={global_step} "
                        f"loss={loss.numpy():.4f} "
                        f"all_finite_grad={bool(all_finite_grad.numpy())} "
                        f"grad_norm={float(grad_norm.numpy()):.4f} "
                    ) 
                    
                    skip_step_count += 1
                    skip_step_count_in_epoch += 1
                    checkpoint.skip_step_count_in_epoch.assign(skip_step_count_in_epoch)
            else:
                step_loss = float(loss.numpy())
                total_loss += step_loss
                
            global_step += 1
            checkpoint.step.assign_add(1)
            checkpoint.global_step.assign(global_step)
            checkpoint.total_loss.assign(total_loss)
            
            pgBar.update(1)
            pgBar.set_postfix(
                loss=f"{step_loss:.4f}",
                avg=f"{(total_loss/max(global_step - skip_step_count_in_epoch, 1)):.4f}",
                except_count=skip_step_count_in_epoch,
                shard=shard_idx
            )
            
            if step != 0 and step % 100 == 0:
                manager.save()

        checkpoint.shard_idx.assign_add(1)
        checkpoint.step.assign(0)
        manager.save()
        
    pgBar.close()
    
    if SAVE_RESULT == True:
        #결과 저장
        result.loc[len(result)] = {
            "epoch": epoch + 1,
            "train_avg_loss": (total_loss/(global_step - skip_step_count_in_epoch)),
        }
        result.to_csv(config['training']['result']['path'], encoding='utf-8-sig', index=False)
    
    # epoch 종료 시 lr 감소
    current_lr = float(base_opt.learning_rate.numpy())
    new_lr = max(current_lr * decay_rate, min_lr)
    base_opt.learning_rate.assign(new_lr)
    
    print(f"Epoch {epoch+1} 평균 Loss: {(total_loss/(global_step - skip_step_count_in_epoch)):.4f}")
    print(f"다음 learning_rate: {current_lr} -> {new_lr}")
    checkpoint.epoch.assign_add(1)
    checkpoint.shard_idx.assign(0)
    checkpoint.global_step.assign(0)
    checkpoint.skip_step_count_in_epoch.assign(0)
    checkpoint.total_loss.assign(0)
    sign_model.save_weights(config['model']['save_path'] + f"_{epoch + 1}" + config['model']['ext'])
    manager.save() 

Epoch 1: 100%|██████████| 6/6 [00:11<00:00,  1.97s/step, avg=15.6246, except_count=0, loss=4.4170, shard=5]


Epoch 1 평균 Loss: 15.6246
다음 learning_rate: 9.999999747378752e-06 -> 1e-05


Epoch 2: 100%|██████████| 6/6 [00:04<00:00,  1.31step/s, avg=7.2653, except_count=0, loss=3.8792, shard=5]  


Epoch 2 평균 Loss: 7.2653
다음 learning_rate: 9.999999747378752e-06 -> 1e-05


Epoch 3: 100%|██████████| 6/6 [00:04<00:00,  1.34step/s, avg=4.7518, except_count=0, loss=2.3909, shard=5]


...

Epoch 6 평균 Loss: 0.4159
다음 learning_rate: 9.999999747378752e-06 -> 1e-05


Epoch 7: 100%|██████████| 6/6 [00:04<00:00,  1.26step/s, avg=0.2645, except_count=0, loss=0.0570, shard=5]


Epoch 7 평균 Loss: 0.2645
다음 learning_rate: 9.999999747378752e-06 -> 1e-05


Epoch 8:  50%|█████     | 3/6 [00:02<00:02,  1.32step/s, avg=0.1485, except_count=0, loss=0.1645, shard=2]

KeyboardInterrupt: 

In [ ]:
#sign_model.load_weights(r"./model_weight/sign_model(new_normalize)_15.weights.h5")
sign_model.load_weights(r"./model_weight/sign_model(finetune)_7.weights.h5")

In [9]:
import utils as utils

config = utils.load_config("./configs/config.yaml")

def predict_data(model, data_path, show_report=False, save_result=True):
    if save_result == True:
        result = pd.read_csv(config['training']['result']['path'])
    
    data = pd.read_csv(data_path, encoding='utf-8-sig')
    data.dropna(inplace=True)
    #data = data[(data['match_ratio'] > 0.7)]

    data['morpheme'] = data['morpheme'].apply(
        lambda m: np.array(vocabulary.encode(m), dtype=np.int32)
    )

    #데이터 셔플
    data = data.sample(frac=1, random_state=SEED).reset_index(drop=True)

    dataset = ds.make_dataset_from_df(data, BATCH_SIZE) 

    batch_idx = 0
    total_wer = 0

    edit_count = 0
    word_count = 0
    
    for ((video, keypoint, video_length), morpheme) in dataset:
        logits = model((video, keypoint), training=False)
        
        logits = tf.cast(logits, tf.float32)
        
        wer, edits, words, refs, hyps = utils.wer_from_logits(
            logits=logits,              # [B,T,C]
            labels=morpheme,             # [B,L] (pad=3)
            input_length=video_length,  # [B]
            blank_index=0,
            pad_index=3
        )
        
        batch_idx += 1
        print(f"batch: {batch_idx}")
        
        if show_report:
            report = utils.format_batch_alignment_reports(
                refs, hyps,
                id_to_token=vocabulary.itos,
                include_equal=False,
                max_samples=2
            )
        
        total_wer += wer
        edit_count += edits
        word_count += words
        
        if show_report:
            print(report + '\n')
        else:
            print("Macro_WER", f"{(total_wer / batch_idx):.4f}", 
                "Micro_WER", f"{(edit_count/word_count):.4f}", "WER:", f"{wer:.4f}", "edits:", edits, "words:", words, "\n")
        
    if save_result == True:
        last_idx = result.index[-1]
        result.loc[last_idx, 'val_macro_wer'] = total_wer / batch_idx
        result.loc[last_idx, 'val_micro_wer'] = edit_count / word_count
        result.to_csv(config['training']['result']['path'], encoding='utf-8-sig', index=False)
    
    return batch_idx, (total_wer / batch_idx), (edit_count / word_count)

#data_path = config['data']['path'] + config['data']['test_path']
data_path = r"C:\Users\user\Downloads\수어 영상\1.Training\finetune.csv"

batch_idx, macro_wer, micro_wer = predict_data(sign_model, data_path, save_result=False)
print(f'batch: {batch_idx}, macro_wer: {macro_wer}, micro_wer: {micro_wer}')


batch: 1
Macro_WER 0.0000 Micro_WER 0.0000 WER: 0.0000 edits: 0 words: 8 

batch: 2
Macro_WER 0.0000 Micro_WER 0.0000 WER: 0.0000 edits: 0 words: 2 

batch: 3
Macro_WER 0.0000 Micro_WER 0.0000 WER: 0.0000 edits: 0 words: 2 

batch: 3, macro_wer: 0.0, micro_wer: 0.0


In [6]:
config = utils.load_config("./configs/config.yaml")
sign_model.save_weights(config['model']['save_path'])